# SPARC Example 05: Corpus Survey Breakdown

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

How many galaxies per survey? What quality tiers? This example
loads the flat CSV — the fastest way to get corpus-level statistics.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import csv
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

rows = []
with open('rotation_curve_corpus_v7_flat.csv') as f:
    for r in csv.DictReader(f):
        rows.append(r)

surveys = Counter(r['survey'] for r in rows)
tiers   = Counter(r['quality_tier'] for r in rows)

print(f"Total galaxies: {len(rows)}")
print("\nSurvey breakdown:")
for s, n in sorted(surveys.items(), key=lambda x: -x[1]):
    print(f"  {s:<15} {n:>4} galaxies")
print("\nQuality tiers:")
for t, n in sorted(tiers.items()):
    label = "hand-curated" if t == "1" else "automated pipeline"
    print(f"  Tier {t}: {n} galaxies ({label})")

In [ ]:
COLORS = {'SPARC':'#1f77b4','THINGS':'#ff7f0e',
          'LITTLE_THINGS':'#2ca02c','WALLABY':'#d62728'}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Pie
labels = list(surveys.keys())
sizes  = [surveys[k] for k in labels]
colors = [COLORS.get(k, '#9467bd') for k in labels]
axes[0].pie(sizes, labels=[f'{l}\n({n})' for l, n in zip(labels, sizes)],
            colors=colors, autopct='%1.0f%%', textprops={'fontsize': 9})
axes[0].set_title('Galaxies by Survey', fontsize=11)

# Stacked bar by tier
x = np.arange(len(COLORS))
tier_data = {}
for r in rows:
    tier_data.setdefault(r['survey'], {}).setdefault(r['quality_tier'], 0)
    tier_data[r['survey']][r['quality_tier']] += 1
for i, (t, color) in enumerate([('1','#2166ac'), ('2','#b2182b')]):
    vals = [tier_data.get(s, {}).get(t, 0) for s in COLORS]
    axes[1].bar(x + i*0.35, vals, 0.35, label=f'Tier {t}', color=color, alpha=0.85)
axes[1].set_xticks(x + 0.175)
axes[1].set_xticklabels(list(COLORS.keys()), fontsize=9)
axes[1].set_ylabel('N galaxies')
axes[1].set_title('Quality Tier by Survey', fontsize=11)
axes[1].legend()

plt.suptitle('EPS Research Unified HI Corpus v7.0 — Survey Coverage (N=438)',
             fontsize=11)
plt.tight_layout()
plt.savefig('ex05_survey_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()